## tl;dr

The latest 30-day Binance USDT spot scan found five close-confirmed +200% run-ups: SYN, TLM, DEXE, BICO, and VANRY. A simple pre-entry fragility score combining seven-day Amihud illiquidity, one-day intraday range, and seven-day ATR produced 14.2x lift in the final 60-day test window at the top 2% cutoff. This is a useful alert/ranking factor, not a robust automatic buy signal: the median selected trade remained negative and performance depended on rare winners.

## Context & Methods

The prediction clock closes after UTC day t. Entry is the first four-hour open of t+1. The positive outcome is any high during t+1 through t+14 reaching at least 3x entry. Features use only information observable before entry. The current event inventory covers 30 days; factor validation uses roughly 400 days of history from the current Binance spot universe.

### Key Assumptions

- Current ExchangeInfo omits historically delisted symbols, so survivorship bias remains.
- Daily/high prices do not guarantee executable fills during vertical moves.
- OI is recent-window enrichment only because Binance OI history is short.
- `current_30d_runups.csv` separates close-confirmed moves from wick-only observations.

## Data

In [ ]:
from pathlib import Path
import json
import pandas as pd

REPORT_DIR = Path('reports/binance_200pct_factor_mining_2026-07-18')
summary = json.loads((REPORT_DIR / 'analysis_summary.json').read_text(encoding='utf-8'))
quality = json.loads((REPORT_DIR / 'data_quality.json').read_text(encoding='utf-8'))
runups = pd.read_csv(REPORT_DIR / 'current_30d_runups.csv')
factors = pd.read_csv(REPORT_DIR / 'factor_stability.csv')
strategies = pd.read_csv(REPORT_DIR / 'strategy_backtest_summary.csv')
watchlist = pd.read_csv(REPORT_DIR / 'current_watchlist.csv')
quality['loaded_rows_4h'], quality['duplicate_symbol_time_rows'], quality['invalid_ohlc_rows']

## Results

In [ ]:
runups[[
    'symbol', 'classification', 'close_to_later_high_return',
    'close_entry_time', 'close_peak_time', 'oi_to_cmc_mcap',
    'oi_change_1d', 'funding_bps'
]].sort_values('close_to_later_high_return', ascending=False)

In [ ]:
factors[[
    'factor', 'direction', 'stable', 'train_auc',
    'train_top_decile_lift', 'test_auc', 'test_top_decile_lift'
]].head(12)

In [ ]:
fragility = summary['factor_validation']['model_metrics']['simple_fragility_score']
pd.DataFrame([
    {'period': 'train', **fragility['train']},
    {'period': 'test', **fragility['test']},
])[['period', 'base_rate', 'auc', 'top_2pct_precision', 'top_2pct_lift']]

In [ ]:
strategies[[
    'strategy', 'top_k', 'median_phase_mean_net_return',
    'median_phase_median_net_return', 'median_phase_win_rate',
    'median_phase_hit_3x_rate', 'worst_trade_drawdown'
]].sort_values(['strategy', 'top_k'])

In [ ]:
watchlist[[
    'symbol', 'fragility_score_pctile', 'stage', 'return_7d',
    'volume_ratio_1d_20d', 'oi_to_cmc_mcap', 'oi_change_1d',
    'funding_bps'
]].head(15)

## Takeaways

1. Use the fragility score to reduce the daily universe to roughly the top 2%, not as an unconditional entry.
2. Treat OI growth as optional context. It was not common across the five close-confirmed current run-ups.
3. Exclude recent +200% movers, extreme funding, very high futures/spot volume, and already-chasing price/volume states.
4. If traded at all, use a small lottery sleeve because the median trade and win rate remain weak despite strong tail-event lift.
5. Accumulate new snapshots prospectively before promoting the alert into an automated strategy.